In [1]:
import pandas as pd
from helpers import get_factor, get_price

In [2]:
CDF = pd.read_csv("../production/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

/tmp/ipykernel_1385580/3001582791.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [3]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

/tmp/ipykernel_1385580/2825335293.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
/tmp/ipykernel_1385580/2825335293.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")


In [4]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [5]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [6]:
dataset = CDF2.merge(seem, left_on="variable", right_on="sseid")

In [7]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [8]:
magic2 = magic[["value"]]

In [9]:
#magic2.sort_values(["produced_at", "value"])

In [10]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [11]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")

In [12]:
merged["revenue"] = merged["value"] * merged["price"]
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [13]:
tmp1.reset_index(inplace=True)

In [14]:
revenue = tmp1[["plantid", "revenue"]]

In [17]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [18]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [19]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [20]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [21]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [22]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [25]:
tmp2 = tmp1

In [26]:
#tmp2

In [27]:
coal_cost_per_t = 103.5 or 120
co2_cost = 70
#electricity_price = 78.50

In [28]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [29]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [30]:
co2s3.dtypes

plantid      object
amount_2    float64
dtype: object

In [31]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
1,BB45025564,7.982573e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,14.134000,988.542800,78.280615,-268.566091
12,BWpf-450-2948214-00000000,1.721634e+08,Steinkohle,GKM Block 6,92027.0,2.68,120,3.430000,233.658110,153.582090,-215.076770
0,BB23020490,1.093048e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.015000,211.050000,98.315217,-200.060395
32,NW500-0342658,2.653052e+07,Steinkohle,Scholven 1 DT,48161.0,2.68,120,1.993000,136.138730,89.238806,-198.847014
33,NW500-0915123,1.619746e+08,Steinkohle,Datteln 4,631.0,2.68,120,2.945000,206.105830,131.865672,-175.996855
11,BWpf-450-2797933-00000000,1.932364e+08,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,3.185000,222.950000,142.611940,-172.325495
24,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,16.485000,1153.739930,91.301538,-152.041538
19,NI01241117210,3.552431e+07,Steinkohle,DT30 GuD Süd Wolfsburg,0.0,2.68,120,1.429000,100.030000,63.985075,-128.490761
17,MV30000226,7.126096e+07,Steinkohle,Kraftwerk Rostock Block A,14468.0,2.68,120,1.728000,119.947240,77.373134,-126.059416
2,BB45025611,6.055226e+08,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,9.713000,665.361830,53.795077,-113.634263


In [32]:
#tmp2["profit_adj"] = (tmp2["revenue"] * 1.10) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [33]:
tmp2.sort_values("profit")

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
1,BB45025564,7.982573e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,14.134000,988.542800,78.280615,-268.566091
12,BWpf-450-2948214-00000000,1.721634e+08,Steinkohle,GKM Block 6,92027.0,2.68,120,3.430000,233.658110,153.582090,-215.076770
0,BB23020490,1.093048e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.015000,211.050000,98.315217,-200.060395
32,NW500-0342658,2.653052e+07,Steinkohle,Scholven 1 DT,48161.0,2.68,120,1.993000,136.138730,89.238806,-198.847014
33,NW500-0915123,1.619746e+08,Steinkohle,Datteln 4,631.0,2.68,120,2.945000,206.105830,131.865672,-175.996855
11,BWpf-450-2797933-00000000,1.932364e+08,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,3.185000,222.950000,142.611940,-172.325495
24,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,16.485000,1153.739930,91.301538,-152.041538
19,NI01241117210,3.552431e+07,Steinkohle,DT30 GuD Süd Wolfsburg,0.0,2.68,120,1.429000,100.030000,63.985075,-128.490761
17,MV30000226,7.126096e+07,Steinkohle,Kraftwerk Rostock Block A,14468.0,2.68,120,1.728000,119.947240,77.373134,-126.059416
2,BB45025611,6.055226e+08,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,9.713000,665.361830,53.795077,-113.634263
